# 03 · Filter & Rank — run the shared multi-layer filter (oligomer)

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25
projects use. **For Project 04** you run it with `design_type="oligomer"` so the cutoffs match
symmetric assemblies (subunit scRMSD, pLDDT, interface pAE), then add a **symmetry-RMSD** screen
on top (does it close into the intended order?).

Run `00`–`02` first so `results/assemblies.csv` exists.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline
This is the cohort's shared module — the `oligomer` cutoffs live in its `DEFAULT_CUTOFFS`.

In [ ]:
import filtering_pipeline as fp
import pandas as pd

print("oligomer cutoffs:", fp.DEFAULT_CUTOFFS["oligomer"])

## 1 · Predict each assembly (AF2-Multimer) and attach metrics

For each row in the pool, run `multimer_predict` to get subunit scRMSD, interface pAE, symmetry
RMSD, pLDDT, and an interface-energy proxy. Mock backend here (deterministic, SYNTHETIC); switch
to `tool="af2"` on an A100. We keep `predicted_order` so notebook 04 can study wrong-oligomer risk.

In [ ]:
from sym_tools import multimer_predict

pool = pd.read_csv("results/assemblies.csv")
PRED_TOOL = "mock"   # -> "af2" on an A100

pred_rows = []
for _, r in pool.iterrows():
    s = multimer_predict(r["sequence"], r["symmetry"], tool=PRED_TOOL)
    pred_rows.append(dict(
        assembly_id=r["assembly_id"], seq_index=r["seq_index"], symmetry=r["symmetry"],
        n_subunits=r["n_subunits"], tied=r["tied"],
        subunit_scrmsd=s.subunit_scrmsd, interface_pae=s.interface_pae,
        symmetry_rmsd=s.symmetry_rmsd, plddt=s.plddt,
        interface_energy=s.interface_energy, predicted_order=s.predicted_order))
pred = pd.DataFrame(pred_rows)
pred.to_csv("results/predictions.csv", index=False)
print("wrote results/predictions.csv", pred.shape)
pred.head()

## 2 · Build `fp.Design` objects (design_type='oligomer')

The filter operates on `fp.Design` records. Map the multimer metrics onto its fields:
`scrmsd` ← subunit scRMSD, `pae_interaction` ← interface pAE, `plddt` ← assembly pLDDT,
`rosetta_dG` ← interface-energy proxy. The symmetry RMSD has no native field, so we carry it in
`extra` and apply it as an extra screen after the standard layers.

In [ ]:
designs = []
for _, r in pred.iterrows():
    designs.append(fp.Design(
        design_id=f"{r['assembly_id']}#{int(r['seq_index'])}",
        sequence="",                         # not needed for the confidence layers
        design_type="oligomer",
        scrmsd=r["subunit_scrmsd"],
        plddt=r["plddt"],
        pae_interaction=r["interface_pae"],
        rosetta_dG=r["interface_energy"],    # interface-energy proxy (more negative = better)
        extra={"symmetry": r["symmetry"], "n_subunits": int(r["n_subunits"]),
               "tied": bool(r["tied"]), "symmetry_rmsd": r["symmetry_rmsd"],
               "predicted_order": int(r["predicted_order"])},
    ))
print(len(designs), "Design objects built (design_type='oligomer')")

## 3 · Run the shared pipeline

`run_pipeline(..., design_type="oligomer")` applies the layers in order with the oligomer
cutoffs and returns a ranked DataFrame. We use layers 1 (self-consistency: subunit scRMSD +
pLDDT + interface pAE) and 3 (physics: interface energy). Layer 2 (orthogonal) needs a second
predictor's scRMSD — wire it in when you have one.

In [ ]:
df_ranked = fp.run_pipeline(designs, design_type="oligomer", use_layers=(1, 3))
df_ranked.to_csv("results/ranked.csv", index=False)
top = fp.report(df_ranked, top_n=10, save_prefix="results/proj04")
top

## 4 · Extra screen — symmetry RMSD (does it close into the target order?)

The standard layers check the subunit and the interface; the **symmetry RMSD** asks whether the
whole assembly closes into the intended point group. A design can pass interface pAE yet fail
here (confident interface, wrong global arrangement). Apply it as an explicit gate and report
how many survivors it removes.

In [ ]:
import numpy as np
SYM_RMSD_CUTOFF = 2.0   # Å; project-specific — calibrate against your reference homo-oligomers

# Pull the symmetry-RMSD and symmetry label out of `extra` onto the ranked frame.
df_ranked["symmetry_rmsd"] = df_ranked["extra"].apply(
    lambda e: e.get("symmetry_rmsd") if isinstance(e, dict) else np.nan)
df_ranked["symmetry"] = df_ranked["extra"].apply(
    lambda e: e.get("symmetry") if isinstance(e, dict) else None)
passed_layers = df_ranked[df_ranked["layers_passed"] >= 1]
sym_clean = passed_layers[passed_layers["symmetry_rmsd"] <= SYM_RMSD_CUTOFF]
print(f"passed standard layers: {len(passed_layers)}")
print(f"of those, symmetry-RMSD-clean (<= {SYM_RMSD_CUTOFF} Å): {len(sym_clean)}")
print("  -> the gap is designs with a confident interface but the WRONG global arrangement.")

## 5 · Survival-at-each-layer + assembly-success rate (honest accounting)

Report how many designs pass each layer, **per symmetry**. The assembly-success rate is the
fraction passing the full filter *including* the symmetry-RMSD screen. Expect it to be modest
and to differ by symmetry order — report it, don't hide it.

In [ ]:
# Per-symmetry success accounting (mock numbers are SYNTHETIC EXAMPLE_DATA).
# (df_ranked["symmetry"] and sym_clean were derived in the symmetry-RMSD cell above.)
gen = df_ranked.groupby("symmetry").size().rename("generated")
passed = df_ranked[df_ranked["layers_passed"] >= 1].groupby("symmetry").size().rename("passed_layers")
clean = sym_clean.groupby("symmetry").size().rename("symmetry_clean")
acct = pd.concat([gen, passed, clean], axis=1).fillna(0).astype(int)
acct["success_rate"] = (acct["symmetry_clean"] / acct["generated"]).round(3)
print("assembly-success accounting per symmetry (EXAMPLE_DATA on the mock backend):")
print(acct)

## D3 (part 1) checklist
- [ ] `results/ranked.csv` produced by the **shared** module with `design_type="oligomer"`.
- [ ] Symmetry-RMSD screen applied on top; symmetry-clean survivors reported.
- [ ] Survival-at-each-layer + per-symmetry assembly-success rate reported (N pass / N generated).
- [ ] Mapping assumptions (which metric → which `Design` field) written down.

**Next:** `04_validate.ipynb` — symmetry-order-vs-success, tied-vs-untied, wrong-oligomer analysis.